<a href="https://colab.research.google.com/github/vishaljoshi24/DungeonsAndDragonsTurnClassification/blob/Codex-Branch/prompt_generation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import sys
print(sys.version)
print(sys.executable)

!{sys.executable} -m pip --version

3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
/usr/bin/python3
pip 24.1.2 from /usr/local/lib/python3.12/dist-packages/pip (python 3.12)


In [ ]:
!pip install dspy==3.2.1

In [2]:
!git clone -b Codex-Branch https://github.com/vishaljoshi24/DungeonsAndDragonsTurnClassification/
%cd /content/DungeonsAndDragonsTurnClassification

fatal: destination path 'DungeonsAndDragonsTurnClassification' already exists and is not an empty directory.
/content/DungeonsAndDragonsTurnClassification


In [3]:
import requests

response = requests.get("http://127.0.0.1:11434/api/tags", timeout=10)
print(response.status_code)
print(response.json())

200
{'models': [{'name': 'qwen3:4b', 'model': 'qwen3:4b', 'modified_at': '2026-11-18T11:19:55.688581488Z', 'size': 2497293931, 'digest': '359d7dd4bcdab3d86b87d73ac27966f4dbb9f5efdfcc75d34a8764a09474fae7', 'details': {'parent_model': '', 'format': 'gguf', 'family': 'qwen3', 'families': ['qwen3'], 'parameter_size': '4.0B', 'quantization_level': 'Q4_K_M', 'context_length': 262144, 'embedding_length': 2560, 'runner': 'ggml'}, 'capabilities': ['completion', 'tools', 'thinking']}]}


In [4]:
import subprocess

result = subprocess.run(
    ["ollama", "pull", "qwen3:4b"],
    capture_output=False,
    text=True,
)

print("Download finished with exit code:", result.returncode)

Download finished with exit code: 0


In [5]:
import requests

response = requests.get("http://127.0.0.1:11434/api/tags", timeout=10)
response.raise_for_status()

models = response.json()["models"]
for model in models:
    print(model["name"])


qwen3:4b


In [7]:
import pandas as pd
import dspy

In [8]:
generation_df = pd.read_excel('Articulation Work Definitions.xlsx')

In [9]:
generation_df

,Category,Definition
0,question game-state,"Players request information about the past, cu..."
1,assert game-state,"Players state, clarify, correct or report info..."
2,acknowledge,"Players confirm, recognise or accept informati..."
3,probe intention,Players question another player's intent to ac...
4,propose intention or action,Players suggest or negotiate a future action o...
5,interact with resource,"Players request, offer, exchange or use a reso..."
6,act on game-state,Players will roleplay or narrate an action tha...
7,argue,"Players explicitly challenge, dispute or disag..."
8,out-of-gameplay,Trivial comments or reactions regarding the ga...


In [10]:
# generation_df.drop(columns=['Jack\'s Codes'], inplace=True)

In [11]:
# generation_df.drop(columns=['Vishal\'s Codes'], inplace=True)

In [12]:
# generation_df.drop(columns=['Notes'], inplace=True)

In [13]:
generation_df

,Category,Definition
0,question game-state,"Players request information about the past, cu..."
1,assert game-state,"Players state, clarify, correct or report info..."
2,acknowledge,"Players confirm, recognise or accept informati..."
3,probe intention,Players question another player's intent to ac...
4,propose intention or action,Players suggest or negotiate a future action o...
5,interact with resource,"Players request, offer, exchange or use a reso..."
6,act on game-state,Players will roleplay or narrate an action tha...
7,argue,"Players explicitly challenge, dispute or disag..."
8,out-of-gameplay,Trivial comments or reactions regarding the ga...


In [14]:
genset = []

for category, definition in generation_df.values:
    examples = dspy.Example(context=category, question=definition).with_inputs("context", "question")
    genset.append(examples)

In [15]:
genset

[Example({'context': 'question game-state', 'question': 'Players request information about the past, current or future game state to update their mental model of the game state. '}) (input_keys={'question', 'context'}),
 Example({'context': 'assert game-state ', 'question': 'Players state, clarify, correct or report information about the past and current game state.'}) (input_keys={'question', 'context'}),
 Example({'context': 'acknowledge', 'question': 'Players confirm, recognise or accept information provided by another player, whilst maintaining the flow of the conversation.'}) (input_keys={'question', 'context'}),
 Example({'context': 'probe intention', 'question': "Players question another player's intent to act, their preferences, plans or willingness."}) (input_keys={'question', 'context'}),
 Example({'context': 'propose intention or action', 'question': 'Players suggest or negotiate a future action or intention for themselves or others. '}) (input_keys={'question', 'context'}),

In [16]:
lm = dspy.LM('ollama_chat/qwen3:4b', api_base = 'http://localhost:11434', api_key='', max_tokens=8192, cache=False)
dspy.configure(lm=lm)

In [17]:
from typing import Literal


class TurnClassifier(dspy.Signature):
    """Given the context for a Dungeons & Dragons game turn and the game turn itself, classify the turn."""
    context: str = dspy.InputField(desc = "The three previous game turns which describe a player's action or their dialogue.")
    question: str = dspy.InputField (desc="The current turn taken by a player, which can include a description of an action or a piece of dialogue.")
    response: Literal['question game-state',
                      'assert game-state',
                      'act on game-state',
                      'propose intention or action',
                      'argue',
                      'acknowledge',
                      'probe intention',
                      'interact with resource',
                      'out-of-gameplay',
                      ] = dspy.OutputField()

class PlayerInstruction(dspy.Signature):
  """Given the category of articulation work and its definition, generate a prompt instruction for how to conduct that type of articulation work."""
  context: str = dspy.InputField(desc='Category of articulation work carried out in D&D')
  question:str = dspy.InputField(desc='Definition of the type of articulation work.')
  player_instruction:str = dspy.OutputField(desc="instruction on how to behave within a D&D game based on the definition of the type of articulation work.")

In [18]:
class PromptGenerator(dspy.Module):
  def __init__(self):
    # self.classifier = classify
    self.generator = dspy.ChainOfThought(PlayerInstruction, caching=False)

  def forward(self, context, question, **kwargs):
    # pred_category = classify(context=context, question=question).response
    prompt = self.generator(
        context=context,
        question=question
    )
    return prompt

prompt_generator = PromptGenerator()


In [19]:
prompt_generator(
    context=genset[2]['context'],
    question=genset[2]['question']
)

Prediction(
    reasoning="In D&D, acknowledging information requires players to confirm understanding without disrupting the narrative flow. This ensures responses are concise and maintain the conversation's momentum while respecting the other player's input.",
    player_instruction='When another player shares information, respond with a brief confirmation such as "Understood," "I see," or "That makes sense" to acknowledge the information while keeping the conversation flowing smoothly.'
)

In [ ]:
generated_prompt = []

for i in range(len(genset)):
    generated_prompt.append(prompt_generator(genset[i]['context'], genset[i]['question']))

In [ ]:
generated_prompt

In [ ]:
generated_prompt[0]['player_instruction']

In [ ]:
genset[0]

In [ ]:
player_instruction = []
articulation_work=[]

for i in range(len(generated_prompt)):
  articulation_work.append(genset[i]['context'])
  player_instruction.append(generated_prompt[i])


In [ ]:
articulation_work

In [ ]:
player_instruction

In [ ]:
articulation_prompts = {
    category: prediction.player_instruction
    for category, prediction in zip(articulation_work, player_instruction)
}

In [ ]:
articulation_prompts

In [ ]:
instruction_df = pd.DataFrame([articulation_prompts]).T

In [ ]:
instruction_df

In [ ]:
filename = 'instructions.xlsx'
instruction_df.to_excel(filename)